# Examples for extended thinking and prompt injection prevention techniques

Example for extended thinking

In [3]:
import json
import anthropic

import os
from dotenv import load_dotenv
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
model = "claude-sonnet-5-5"

In [28]:
ANALYSIS_SYSTEM = """
You are a contract risk analyst.
Identify the risks in a contract and recommend specific mitigations.
Return a JSON object with:
    - "risk_level": "high"/"medium"/"low"
    - "risks": list of 3 strings (most significant part)
    - "mitigations": list of 3 strings (one per risk, same order)
Return only JSON object.
"""
client =  anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
def analyze_clause(clause: str) -> dict:
    response = client.messages.create(
        model=model,
        max_tokens=2048,
        thinking={"type": "adaptive"},
        output_config={"effort": "high"},
        system=ANALYSIS_SYSTEM,
        messages=[{"role": "user", "content": clause}],
    )

    thinking_text = ""
    answer_text = ""
    for block in response.content:
        if block.type == "thinking":
            thinking_text += block.thinking
        elif block.type == "text":
            answer_text += block.text

    thinking_words = len(thinking_text.split())

    result = {}
    raw = answer_text.strip()
    start, end = raw.find("{"), raw.rfind("}") + 1
    if start != -1 and start < end:
        result = json.loads(raw[start:end])

    answer_tokens = 0
    if raw:
        answer_tokens = client.messages.count_tokens(
            model=model,
            messages=[{"role": "user", "content": raw}],
        ).input_tokens

    approx_thinking_tokens = response.usage.output_tokens - answer_tokens

    return {
        "analysis": result,
        "thinking_words": thinking_words,
        "thinking_tokens": approx_thinking_tokens,
    }


In [27]:
print("=== Extended thinking: contract clause risk analysis ===")

clause = (
    "Either party may terminate this agreement immediately upon written notice if the other "
    "party materially breaches any provision and fails to cure such breach within 5 business days "
    "of receiving written notice. Upon termination, all outstanding fees become immediately "
    "due and payable, and the terminated party forfeits any right to refunds or credits."
)

print(f"Clause... {clause[:100]}...\n")

output = analyze_clause(clause=clause)
analysis = output["analysis"]

if analysis:
    print(f"risk analysis: {analysis.get('risk_level')}")
    print(f"risks:")
    for r in analysis.get("risks", []):
        print(f" - {r}")
    print(f"mitigations:")
    for m in analysis.get("mitigations", []):
        print(f" - {m}")

if output['thinking_tokens'] > 0:
    print(f"\n~{output['thinking_tokens']} tokens of internal reasoning visible in the thinking block.")
else:
    print(f"No thinking block created - with adaptive thinking, internal reasoning may not be surfaced even when used.")

=== Extended thinking: contract clause risk analysis ===
Clause... Either party may terminate this agreement immediately upon written notice if the other party materia...

risk analysis: high
risks:
 - Unreasonably short 5-business-day cure period: Complex or systemic breaches (technical, operational, payment-related) are rarely fixable in one week, which exposes the breaching party to sudden termination. The undefined term 'materially breaches any provision' compounds this, since any provision could trigger termination and 'material' is open to dispute.
 - Acceleration of all outstanding fees on termination: The clause makes all outstanding fees immediately due and payable, which could be read to include unearned future fees. This creates large, sudden cash-flow liability and could be punitive or an unenforceable penalty in some jurisdictions.
 - One-sided forfeiture of refunds and credits: The terminated party loses all refunds and credits regardless of the severity of the breach or 